# 09 — Heterogeneous Treatment Effects y Causal ML
**Autores clave:** Athey & Imbens (2016) *Recursive Partitioning* · Wager & Athey (2018) *Causal Forests* · Chernozhukov et al. (2018) *Double/Debiased ML* · Künzel et al. (2019) *Metalearners*

## Del ATE al CATE
Hasta ahora estimamos efectos **promedio**. Pero el efecto puede variar por subpoblación:
$$\tau(x) = E[Y(1) - Y(0) | X = x] \quad \text{(CATE: Conditional Average Treatment Effect)}$$

## Causal Forest (Wager & Athey, 2018)
Extensión de Random Forest para estimar $\tau(x)$:
1. Construir árboles con criterio de **heterogeneidad** (no pureza de clase)
2. Estimar $\hat{\tau}(x)$ como promedio de los efectos en la hoja
3. Inferencia vía **honesty** (partición separada para estimación)

## Double/Debiased ML — DML (Chernozhukov et al., 2018)
Usa ML para controlar flexiblemente los confounders sin sesgo de regularización:
1. Residualizar: $\tilde{Y} = Y - \hat{E}[Y|X]$ y $\tilde{D} = D - \hat{E}[D|X]$
2. Regresar: $\tilde{Y}$ sobre $\tilde{D}$ → estimador Neyman-ortogonal de $\tau$

## S-learner, T-learner, X-learner (Künzel et al., 2019)
Tres estrategias para usar cualquier algoritmo ML como estimador de CATE.

In [ ]:
import os
os.chdir('/Volumes/SSD_Gabo/proyectos/growth-analytics')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import statsmodels.api as sm
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.model_selection import cross_val_predict, KFold
from sklearn.preprocessing import StandardScaler

plt.rcParams.update({
    'figure.dpi': 130, 'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.color': '#e8e8e8', 'axes.axisbelow': True,
    'axes.titlesize': 11, 'axes.titleweight': 'bold', 'legend.frameon': False,
})
np.random.seed(42)

n = 5000

# Covariables
X1 = np.random.normal(0, 1, n)   # edad (estandarizada)
X2 = np.random.normal(0, 1, n)   # educación (estandarizada)
X3 = np.random.binomial(1, 0.5, n).astype(float)  # género

# Propensity score (confounding)
ps = 1 / (1 + np.exp(-(0.3*X1 - 0.4*X2)))
D  = np.random.binomial(1, ps, n).astype(float)

# CATE verdadero: heterogéneo según X1 y X3
# Jóvenes (X1<0) y hombres (X3=0) se benefician más
tau_true = 2.0 + 1.5*X1 - 1.0*X3 + 0.5*X2

# Outcomes potenciales
Y0 = 5.0 + 2.0*X1 + 1.0*X2 - 0.5*X3 + np.random.normal(0, 2, n)
Y1 = Y0 + tau_true
Y  = np.where(D == 1, Y1, Y0)

X_mat = np.column_stack([X1, X2, X3])
df = pd.DataFrame({'Y': Y, 'D': D, 'X1': X1, 'X2': X2, 'X3': X3,
                   'tau_true': tau_true, 'ps': ps})

print(f'ATE verdadero: {tau_true.mean():.4f}')
print(f'ATT verdadero: {df[df["D"]==1]["tau_true"].mean():.4f}')
print(f'Rango de CATE: [{tau_true.min():.2f}, {tau_true.max():.2f}]  — alta heterogeneidad')

## 1 — T-learner y S-learner (Künzel et al., 2019)

In [ ]:
rf_params = dict(n_estimators=200, max_depth=5, random_state=42, n_jobs=-1)

# ── T-learner: modelo separado para tratados y controles ──────────────────
mu1_model = RandomForestRegressor(**rf_params)
mu0_model = RandomForestRegressor(**rf_params)

mu1_model.fit(X_mat[D==1], Y[D==1])
mu0_model.fit(X_mat[D==0], Y[D==0])

tau_tlearner = mu1_model.predict(X_mat) - mu0_model.predict(X_mat)

# ── S-learner: un único modelo con D como feature ─────────────────────────
X_s = np.column_stack([X_mat, D])
s_model = RandomForestRegressor(**rf_params)
s_model.fit(X_s, Y)

X_s1 = np.column_stack([X_mat, np.ones(n)])  # D=1 para todos
X_s0 = np.column_stack([X_mat, np.zeros(n)]) # D=0 para todos
tau_slearner = s_model.predict(X_s1) - s_model.predict(X_s0)

# Correlación con CATE verdadero
r_t = np.corrcoef(tau_true, tau_tlearner)[0,1]
r_s = np.corrcoef(tau_true, tau_slearner)[0,1]

print('T-learner vs S-learner — Künzel et al. (2019)')
print(f'  T-learner: ATE={tau_tlearner.mean():.4f}  corr(CATE_true)={r_t:.4f}')
print(f'  S-learner: ATE={tau_slearner.mean():.4f}  corr(CATE_true)={r_s:.4f}')
print(f'  ATE verdadero: {tau_true.mean():.4f}')

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, (tau_hat, name, r) in zip(axes, [
    (tau_tlearner, 'T-learner', r_t),
    (tau_slearner, 'S-learner', r_s),
]):
    ax.scatter(tau_true[:500], tau_hat[:500], s=8, alpha=0.3, color='#4361ee')
    mn = min(tau_true.min(), tau_hat.min())
    mx = max(tau_true.max(), tau_hat.max())
    ax.plot([mn,mx],[mn,mx],'k--',linewidth=1.5,label='45°')
    ax.set_xlabel('CATE verdadero'); ax.set_ylabel('CATE estimado')
    ax.set_title(f'{name} (r={r:.3f})')
    ax.legend(fontsize=8)

plt.suptitle('CATE verdadero vs estimado — Künzel et al. (2019)', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.show()

## 2 — Double/Debiased ML — DML (Chernozhukov et al., 2018)

El paso de cross-fitting elimina el sesgo de regularización que introduce ML al estimar los nuisance parameters.

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)

Y_resid = np.zeros(n)
D_resid = np.zeros(n)

# Cross-fitting: para cada fold, estimar los nuisance con los otros folds
for train_idx, val_idx in kf.split(X_mat):
    # Modelo de outcome E[Y|X]
    rf_y = RandomForestRegressor(n_estimators=100, max_depth=5, random_state=42)
    rf_y.fit(X_mat[train_idx], Y[train_idx])
    Y_resid[val_idx] = Y[val_idx] - rf_y.predict(X_mat[val_idx])

    # Modelo de tratamiento E[D|X] (propensity)
    lr_d = LogisticRegression(C=1.0, random_state=42)
    lr_d.fit(X_mat[train_idx], D[train_idx])
    D_resid[val_idx] = D[val_idx] - lr_d.predict_proba(X_mat[val_idx])[:, 1]

# Estimación de θ: Y_resid = θ * D_resid + ε
ate_dml = np.sum(D_resid * Y_resid) / np.sum(D_resid**2)

# SE via sandwich
psi = D_resid * (Y_resid - ate_dml * D_resid)
V   = np.mean(D_resid**2)**(-2) * np.mean(psi**2) / n
se_dml = np.sqrt(V)

# Comparación con OLS naive
ate_ols_naive = sm.OLS(Y, sm.add_constant(D)).fit().params['x1']
ate_ols_x     = sm.OLS(Y, sm.add_constant(np.column_stack([D, X_mat]))).fit().params[1]

print('Double/Debiased ML — Chernozhukov et al. (2018)')
print('─' * 55)
print(f'ATE verdadero:          {tau_true.mean():.4f}')
print(f'OLS sin controles:      {ate_ols_naive:.4f}  ← sesgado')
print(f'OLS con controles:      {ate_ols_x:.4f}')
print(f'DML (cross-fit):        {ate_dml:.4f}  SE={se_dml:.4f}  IC95=[{ate_dml-1.96*se_dml:.4f}, {ate_dml+1.96*se_dml:.4f}]')
print()
print('DML es Neyman-ortogonal: el sesgo en E[Y|X] o E[D|X] entra solo en segundo orden.')

## 3 — Visualización de heterogeneidad: CATE por subgrupos

In [ ]:
# Usar T-learner CATE para explorar heterogeneidad
df['tau_tlearner'] = tau_tlearner

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Por cuartiles de X1
df['X1_quartile'] = pd.qcut(df['X1'], q=4, labels=['Q1', 'Q2', 'Q3', 'Q4'])
cate_x1 = df.groupby('X1_quartile', observed=True)[['tau_true', 'tau_tlearner']].mean()
x_pos = np.arange(4)
axes[0].bar(x_pos - 0.2, cate_x1['tau_true'],     0.35, color='#4361ee', alpha=0.8, label='CATE verdadero')
axes[0].bar(x_pos + 0.2, cate_x1['tau_tlearner'], 0.35, color='#f72585', alpha=0.8, label='T-learner')
axes[0].set_xticks(x_pos); axes[0].set_xticklabels(cate_x1.index)
axes[0].set_xlabel('Cuartil X1 (edad)'); axes[0].set_ylabel('CATE')
axes[0].set_title('CATE por Cuartil de X1')
axes[0].legend(fontsize=8)

# Por X3 (género)
cate_x3 = df.groupby('X3')[['tau_true', 'tau_tlearner']].mean()
axes[1].bar([0, 1], cate_x3['tau_true'],     0.35, color='#4361ee', alpha=0.8, label='CATE verdadero')
axes[1].bar([0.35, 1.35], cate_x3['tau_tlearner'], 0.35, color='#f72585', alpha=0.8, label='T-learner')
axes[1].set_xticks([0.175, 1.175]); axes[1].set_xticklabels(['Hombre (0)', 'Mujer (1)'])
axes[1].set_ylabel('CATE'); axes[1].set_title('CATE por Género')
axes[1].legend(fontsize=8)

# Distribución del CATE estimado
axes[2].hist(df['tau_true'],     bins=50, color='#4361ee', alpha=0.6, density=True, label='Verdadero')
axes[2].hist(df['tau_tlearner'], bins=50, color='#f72585', alpha=0.6, density=True, label='T-learner')
axes[2].set_xlabel('CATE')
axes[2].set_title('Distribución del CATE\nVerdadero vs Estimado')
axes[2].legend(fontsize=8)

plt.suptitle('Heterogeneidad del Efecto de Tratamiento (CATE)', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.show()

print('Subgrupos con mayor beneficio (CATE alto):')
high_effect = df.nlargest(int(n*0.1), 'tau_tlearner')
print(f'  Top 10%: X1 media={high_effect["X1"].mean():.3f}  X3 media={high_effect["X3"].mean():.3f}')
print(f'  CATE medio top 10%: verdadero={high_effect["tau_true"].mean():.3f}  estimado={high_effect["tau_tlearner"].mean():.3f}')

## Resumen — La progresión de Econometría Clásica a Causal ML

| Método | Estima | Ventaja | Limitación |
|---|---|---|---|
| OLS | ATE (lineal) | Simple, interpretable | Sesgo, asume linealidad |
| IV / 2SLS | LATE | Trata endogeneidad | Requiere instrumento válido |
| RDD | LATE local | Cuasi-experimento nítido | Validez solo cerca del umbral |
| DiD | ATT | Sin aleatorización | Tendencias paralelas |
| Matching / IPW | ATT / ATE | Flexible | CIA no testeable |
| T-learner / X-learner | CATE | Captura HTE | Sesgo de regularización |
| DML | ATE ortogonal | Robusto a nuisance | Mayor complejidad |
| Causal Forest | CATE puntual + IC | Inferencia honesta | Caja negra |

**Referencias:**
- Athey, S. & Imbens, G.W. (2016). Recursive partitioning for heterogeneous causal effects. *PNAS*, 113(27).
- Wager, S. & Athey, S. (2018). Estimation and inference of heterogeneous treatment effects using random forests. *JASA*, 113(523).
- Chernozhukov, V., Chetverikov, D., Demirer, M., Duflo, E., Hansen, C., Newey, W. & Robins, J. (2018). Double/debiased machine learning for treatment and structural parameters. *The Econometrics Journal*, 21(1).
- Künzel, S.R., Sekhon, J.S., Bickel, P.J. & Yu, B. (2019). Metalearners for estimating heterogeneous treatment effects. *PNAS*, 116(10).